# 10 · Lista de casa: LLMs como componente

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código. **Use GPU.**
- As respostas do modelo variam. O que se avalia é o **método**: você mediu? validou? conferiu à mão? relatou o que viu, inclusive o que deu errado?
- Cerca de metade da nota está nas respostas escritas. Use os números e os exemplos que você obteve.
- Antes de entregar, use **Reiniciar sessão e executar tudo**. As conferências manuais (questões 2 e 3) devem corresponder à execução entregue.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Três versões de um *prompt*, medidas | 2,5 |
| 2 | Extração estruturada com citação conferida | 2,5 |
| 3 | RAG conferido à mão | 2,0 |
| 4 | Ataque e defesa | 2,0 |
| 5 | A conta do custo | 1,0 |

In [ ]:
import importlib.util
import subprocess
import sys

faltando = [p for p, modulo in [("transformers", "transformers"), ("sentence-transformers", "sentence_transformers"), ("pydantic", "pydantic")] if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

import json
import re
import time
import warnings
from pathlib import Path
from typing import Literal, Optional
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEMENTE = 42
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"
MODELO_FRASES = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizador = AutoTokenizer.from_pretrained(MODELO_LLM)
modelo = AutoModelForCausalLM.from_pretrained(MODELO_LLM)
modelo = modelo.to(dispositivo, dtype=torch.float16 if dispositivo == "cuda" else torch.float32).eval()


@torch.no_grad()
def gerar(mensagens, max_tokens=200, temperatura=0.0):
    """Envia a conversa ao modelo e devolve apenas o texto da resposta (a mesma função da aula)."""
    texto = tokenizador.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entradas = tokenizador(texto, return_tensors="pt").to(dispositivo)
    opcoes = {"do_sample": True, "temperature": temperatura} if temperatura > 0 else {"do_sample": False}
    saida = modelo.generate(**entradas, max_new_tokens=max_tokens, pad_token_id=tokenizador.pad_token_id or tokenizador.eos_token_id, **opcoes)
    return tokenizador.decode(saida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def interpretar(resposta):
    """1 = não recomenda, 0 = recomenda, None = resposta fora do formato."""
    limpa = resposta.strip().upper().replace("Ã", "A")
    return 1 if limpa.startswith("NAO") else 0 if limpa.startswith("SIM") else None


def extrair_json(texto):
    achado = re.search(r"\{.*\}", texto, flags=re.DOTALL)
    if not achado:
        return None
    try:
        return json.loads(achado.group(0))
    except json.JSONDecodeError:
        return None


URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend", "review_text"])
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"]
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)

N_REFERENCIA = 60 if dispositivo == "cuda" else 10
curtas = df[df["texto"].str.len().between(40, 400)]
referencia = pd.concat([
    curtas[curtas["nao_recomenda"] == classe].sample(N_REFERENCIA // 2, random_state=7) for classe in (0, 1)       # semente diferente da aula
]).sample(frac=1, random_state=7).reset_index(drop=True)[["texto", "nao_recomenda"]]

print(f"dispositivo: {dispositivo} · conjunto de referência: {len(referencia)} avaliações")

## Questão 1 (2,5 pontos): três versões de um *prompt*, medidas

a) Traga para cá a sua função `avaliar(classificar, textos, rotulos)` do exercício 3 da aula. Ela devolve um dicionário com `n`, `invalidas`, `acuracia` (respostas inválidas contam como erro) e `segundos_por_item`.

b) O dicionário `PROMPTS_Q1` já tem a versão `"v1"`, propositalmente pobre. Acrescente as versões `"v2"` e `"v3"`, cada uma mudando **uma coisa** em relação à anterior (por exemplo: descrever melhor a tarefa; fixar o formato da resposta; dar critérios para os casos difíceis). As três devem pedir a resposta `SIM` (recomenda) ou `NAO` (não recomenda).

c) Monte `tabela_q1`: um DataFrame com uma linha por versão (índice `"v1"`, `"v2"`, `"v3"`) e as quatro colunas devolvidas por `avaliar`, medindo no conjunto `referencia`.

In [ ]:
# ✏️ Questão 1 (a)
def avaliar(classificar, textos, rotulos):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✏️ Questão 1 (b, c)
PROMPTS_Q1 = {
    "v1": "O cliente gostou? SIM ou NAO.",
}
# SEU CÓDIGO AQUI
PROMPTS_Q1["v2"] = ...
PROMPTS_Q1["v3"] = ...

tabela_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
_falso = {"a": 1, "b": 0, "c": None, "d": 1, "e": None}.get
_r = avaliar(_falso, ["a", "b", "c", "d", "e"], [1, 1, 0, 1, 0])
assert _r["n"] == 5 and _r["invalidas"] == 2 and abs(_r["acuracia"] - 0.4) < 1e-9, "a função avaliar não passou no teste com o classificador de mentira"
assert list(PROMPTS_Q1) == ["v1", "v2", "v3"] and len(set(PROMPTS_Q1.values())) == 3 and all("SIM" in p and "NAO" in p for p in PROMPTS_Q1.values())
assert list(tabela_q1.index) == ["v1", "v2", "v3"] and set(tabela_q1.columns) == {"n", "invalidas", "acuracia", "segundos_por_item"}
assert (tabela_q1["n"] == len(referencia)).all() and tabela_q1["acuracia"].between(0, 1).all()
_erro_padrao = float(np.sqrt(0.25 / len(referencia)))
print(f"Formato correto. Para referência: com {len(referencia)} exemplos, o erro padrão de uma acurácia perto de 50% é de {_erro_padrao:.1%}; perto de 90%, de {np.sqrt(0.09 / len(referencia)):.1%}.")

**1d)** O que você mudou da v1 para a v2, e da v2 para a v3? O que esperava, e o que a tabela mostra (acurácia, respostas inválidas, tempo)?

**1e)** As diferenças entre as versões são maiores que a incerteza de uma medição com este número de exemplos (veja o erro padrão impresso acima)? Você escolheu o *prompt* olhando para este conjunto: que cuidado isso exige antes de anunciar a acurácia do componente?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,5 pontos): extração estruturada com citação conferida

Queremos transformar avaliações negativas em registros. Para reduzir invenções, cada registro deve trazer uma **citação literal** do texto que justifique a classificação, e o programa vai **conferir** se a citação existe mesmo.

a) Defina a classe `Reclamacao` (Pydantic) com: `problema`, um entre `"não entregue"`, `"atraso"`, `"defeito"`, `"diferente do anunciado"`, `"atendimento"`, `"outro"`; `gravidade`, inteiro de 1 a 3; `citacao`, texto com pelo menos 5 caracteres.

b) Escreva `citacao_confere(citacao, texto)`: devolve `True` se a citação aparece no texto, ignorando diferenças de maiúsculas e de espaços repetidos (normalize os dois com `" ".join(s.lower().split())`).

c) Escreva `extrair(texto)`: monta o *prompt*, chama `gerar`, extrai o JSON (`extrair_json`), valida com `Reclamacao` e confere a citação. Devolve a `Reclamacao`, ou `None` se qualquer etapa falhar.

d) Aplique `extrair` às avaliações de `negativas` e monte `tabela_q2`, com as colunas `texto`, `valida` (booleano), `problema`, `gravidade` e `citacao` (as três últimas vazias quando não houver resposta válida).

In [ ]:
N_Q2 = 15 if dispositivo == "cuda" else 3
negativas = curtas[curtas["nao_recomenda"] == 1].sample(N_Q2, random_state=11)["texto"].tolist()

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
class Reclamacao(BaseModel):
    ...

def citacao_confere(citacao, texto):
    ...

def extrair(texto):
    ...

tabela_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert Reclamacao.model_validate({"problema": "defeito", "gravidade": 3, "citacao": "veio quebrado"}).gravidade == 3
for _invalido in [{"problema": "frete", "gravidade": 2, "citacao": "abcdef"}, {"problema": "atraso", "gravidade": 4, "citacao": "abcdef"},
                  {"problema": "atraso", "gravidade": 2, "citacao": "abc"}, {"problema": "atraso", "gravidade": 2}]:
    try:
        Reclamacao.model_validate(_invalido)
        raise AssertionError(f"o esquema deveria recusar {_invalido}")
    except ValidationError:
        pass
assert citacao_confere("Veio   QUEBRADO", "O produto veio quebrado e sujo.") and not citacao_confere("veio arranhado", "O produto veio quebrado e sujo.")
assert list(tabela_q2.columns) == ["texto", "valida", "problema", "gravidade", "citacao"] and len(tabela_q2) == len(negativas)
assert tabela_q2["valida"].dtype == bool and all(citacao_confere(c, t) for c, t in tabela_q2.loc[tabela_q2["valida"], ["citacao", "texto"]].itertuples(index=False))
print(f"Formato correto. Respostas válidas: {int(tabela_q2['valida'].sum())} de {len(tabela_q2)}.")

e) **Conferência manual.** Leia as respostas **válidas** (até 8) e registre em `corretas_q2` uma lista de booleanos, na ordem em que aparecem em `tabela_q2[tabela_q2["valida"]]`: o `problema` escolhido está correto para aquela avaliação?

In [ ]:
# ✏️ Questão 2 (e): preencha DEPOIS de ler as respostas
pd.set_option("display.max_colwidth", 200)
print(tabela_q2[tabela_q2["valida"]].head(8).to_string())
# SEU CÓDIGO AQUI
corretas_q2 = [...]

In [ ]:
# ✅ Conferência da questão 2 (e) (não altere)
assert len(corretas_q2) == min(8, int(tabela_q2["valida"].sum())) and all(isinstance(c, bool) for c in corretas_q2)
print(f"Das respostas válidas conferidas, você considerou corretas {sum(corretas_q2)} de {len(corretas_q2)}.")

**2f)** Quantas respostas foram descartadas, e em que etapa (JSON, esquema, citação)? Entre as válidas, quantas estavam corretas? Dê um exemplo de resposta **válida e errada**, se houver. O que a exigência de citação literal conferida por código garante, e o que ela não garante?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,0 pontos): RAG conferido à mão

O acervo desta questão são as avaliações do **segundo** produto mais avaliado da base.

a) Calcule `vetores_q3` (vetores normalizados do acervo) e escreva `responder(pergunta, k=5)`, que recupera os `k` trechos mais parecidos e gera a resposta com um *prompt* que exija citações numeradas e a resposta `Não sei.` quando faltar informação. A função devolve uma tupla `(resposta, trechos)`.

b) Escreva três perguntas em `perguntas_q3`: duas que as avaliações devem conseguir responder e uma que, com certeza, **não** conseguem. Execute a célula de resultados.

c) Para cada pergunta, registre em `conferencia_q3` um dicionário com duas chaves: `"recuperacao"` (os trechos recuperados tratam da pergunta? `True`/`False`) e `"resposta"` (a resposta está apoiada nos trechos citados, ou diz corretamente que não sabe? `True`/`False`).

In [ ]:
segundo = df["product_id"].value_counts().index[1]
acervo_q3 = df.loc[df["product_id"] == segundo, "texto"].tolist()
print(f"Produto: {df.loc[df['product_id'] == segundo, 'product_name'].iloc[0]} · {len(acervo_q3)} avaliações")
modelo_frases = SentenceTransformer(MODELO_FRASES, device=dispositivo)

In [ ]:
# ✏️ Questão 3 (a, b)
# SEU CÓDIGO AQUI
vetores_q3 = ...

def responder(pergunta, k=5):
    ...

perguntas_q3 = [..., ..., ...]

In [ ]:
# Resultados (não altere)
for pergunta in perguntas_q3:
    resposta, trechos = responder(pergunta)
    print(f"PERGUNTA: {pergunta}\nRESPOSTA: {resposta}")
    for i, trecho in enumerate(trechos, start=1):
        print(f"   [{i}] {trecho[:150]}")
    print()

In [ ]:
# ✏️ Questão 3 (c): preencha DEPOIS de ler os resultados acima
# SEU CÓDIGO AQUI
conferencia_q3 = {
    perguntas_q3[0]: {"recuperacao": ..., "resposta": ...},
    perguntas_q3[1]: {"recuperacao": ..., "resposta": ...},
    perguntas_q3[2]: {"recuperacao": ..., "resposta": ...},
}

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert vetores_q3.shape[0] == len(acervo_q3) and np.allclose(np.linalg.norm(vetores_q3, axis=1), 1, atol=1e-3)
_resposta, _trechos = responder("O produto é bom?", k=3)
assert isinstance(_resposta, str) and len(_trechos) == 3 and all(t in acervo_q3 for t in _trechos), "responder deve devolver (resposta, trechos do acervo)"
assert len(perguntas_q3) == 3 and set(conferencia_q3) == set(perguntas_q3)
assert all(set(v) == {"recuperacao", "resposta"} and all(isinstance(x, bool) for x in v.values()) for v in conferencia_q3.values())
print("Formato correto.")
print(pd.DataFrame(conferencia_q3).T.to_string())

**3d)** Para cada pergunta, justifique o seu julgamento em uma ou duas frases, apontando o trecho que sustenta (ou contradiz) a resposta. Onde houve falha, ela foi da **recuperação** ou da **geração**? O que o modelo fez com a pergunta sem resposta?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): ataque e defesa

a) Escreva em `ataques_q4` **duas** avaliações claramente negativas que tentem, por estratégias diferentes, fazer o classificador responder `SIM` (por exemplo: uma ordem direta; um texto que finge ser o fim da avaliação e o começo de uma nova instrução; um pedido em outro idioma).

b) Escreva `INSTRUCAO_DEFENDIDA_Q4`, uma versão da instrução que tente resistir aos ataques, e a função `mensagens_defendidas(texto)`, que monta a conversa (você pode usar delimitadores em volta do texto).

c) Execute a célula de medição, que aplica os dois ataques à instrução simples e à defendida.

In [ ]:
INSTRUCAO_SIMPLES = (
    "Você classifica avaliações de clientes de uma loja virtual. "
    "Responda com uma única palavra: SIM, se o cliente recomendaria o produto, ou NAO, se não recomendaria."
)

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
ataques_q4 = [..., ...]
INSTRUCAO_DEFENDIDA_Q4 = ...

def mensagens_defendidas(texto):
    ...

In [ ]:
# Medição (não altere): 1 = "não recomenda" (correto), 0 = o ataque funcionou, None = resposta fora do formato
_linhas = []
for i, ataque in enumerate(ataques_q4, start=1):
    simples = interpretar(gerar([{"role": "system", "content": INSTRUCAO_SIMPLES}, {"role": "user", "content": f"Avaliação: {ataque}"}], max_tokens=5))
    defendida = interpretar(gerar(mensagens_defendidas(ataque), max_tokens=5))
    _linhas.append({"ataque": i, "instrução simples": simples, "instrução defendida": defendida})
tabela_q4 = pd.DataFrame(_linhas).set_index("ataque")
tabela_q4

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert len(ataques_q4) == 2 and ataques_q4[0] != ataques_q4[1] and all(isinstance(a, str) and len(a) > 30 for a in ataques_q4)
assert INSTRUCAO_DEFENDIDA_Q4 != INSTRUCAO_SIMPLES and "SIM" in INSTRUCAO_DEFENDIDA_Q4 and "NAO" in INSTRUCAO_DEFENDIDA_Q4
_m = mensagens_defendidas("texto de teste")
assert _m[0] == {"role": "system", "content": INSTRUCAO_DEFENDIDA_Q4} and _m[-1]["role"] == "user" and "texto de teste" in _m[-1]["content"]
print("Formato correto.")

**4d)** Descreva a estratégia de cada ataque e o que a tabela mostra: algum funcionou contra a instrução simples? A defesa mudou o resultado? Por que um resultado favorável nesses dois testes **não** permite dizer que o componente está protegido?

**4e)** Suponha que este classificador decida, sozinho, quais clientes recebem um cupom de desculpas. Proponha duas medidas de **desenho do sistema** (fora do *prompt*) que limitem o dano de uma injeção bem-sucedida.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): a conta do custo

a) Com o tempo por avaliação que você mediu na questão 1 (melhor versão), estime quantas horas de GPU seriam necessárias para classificar **500 mil** avaliações com o LLM. Depois descreva, com números, um plano alternativo em que o LLM classifica apenas uma amostra e um modelo clássico faz o restante: tamanho da amostra, como você conferiria a qualidade dos rótulos, e como mediria o modelo final.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._